<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda:
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [2]:
from functools import lru_cache

import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split


@lru_cache(maxsize=1)
def _fetch_openml_dataset():
    candidates = [
        ("Fashion-MNIST", {"version": 1}),
        ("mnist_784", {}),
    ]

    last_error = None
    for name, extra_kwargs in candidates:
        try:
            X, y = fetch_openml(
                name=name,
                as_frame=False,
                parser="auto",
                return_X_y=True,
                **extra_kwargs,
            )
            return X.astype(np.float32), y.astype(np.int64), name
        except Exception as exc:
            last_error = exc

    raise RuntimeError(f"Nao foi possivel carregar o dataset via OpenML. Ultimo erro: {last_error}")


def load_data(seed=42, test_size=0.2, max_samples=None, normalize=False):
    X, y, _ = _fetch_openml_dataset()

    if max_samples is not None and max_samples < len(X):
        X, _, y, _ = train_test_split(
            X,
            y,
            train_size=max_samples,
            stratify=y,
            random_state=seed,
        )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=seed,
    )

    if normalize:
        X_train = X_train / 255.0
        X_test = X_test / 255.0

    return X_train, X_test, y_train, y_test

**Resposta (Questão 1):**

A normalização não é estritamente necessária para modelos baseados em árvores (como Random Forest e AdaBoost), pois eles são invariantes a transformações de escala nos atributos, uma vez que as divisões nos nós são baseadas em limiares de valores individuais.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [3]:
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier


def train_random_forest(
    X_train,
    y_train,
    seed=42,
    n_estimators=200,
    max_depth=None,
    n_jobs=-1,
    **kwargs,
):

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=seed,
        n_jobs=n_jobs,
        **kwargs,
    )
    model.fit(X_train, y_train)
    return model


def train_adaboost(
    X_train,
    y_train,
    seed=42,
    n_estimators=80,
    learning_rate=0.7,
    base_depth=2,
    **kwargs,
):

    base_estimator = DecisionTreeClassifier(max_depth=base_depth, random_state=seed)

    # Compatibilidade scikit-learn
    model_kwargs = dict(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        random_state=seed,
    )
    model_kwargs.update(kwargs)

    if "estimator" in AdaBoostClassifier().get_params():
        model_kwargs["estimator"] = base_estimator
    else:
        model_kwargs["base_estimator"] = base_estimator

    model = AdaBoostClassifier(**model_kwargs)
    model.fit(X_train, y_train)
    return model

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [4]:
def evaluate(model, X_test, y_test, average="macro", return_metrics=False):

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average=average, zero_division=0
    )

    metrics = {
        "accuracy": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }
    return metrics if return_metrics else metrics["accuracy"]

A função criada recebe como entrada os modelos juntamente com seus respectivos conjuntos de teste (X). A partir das previsões geradas pelo método `predict`, são obtidas as instâncias que compõem o relatório final, permitindo calcular rapidamente as principais métricas de avaliação. Entre elas, destaca-se a análise global do desempenho por meio da acurácia. Adicionalmente, também são consideradas métricas mais específicas, como precisão, *recall* e diferentes variações do f-score.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [ ]:
def run_pipeline(model_type="rf", seed=42, normalize=False, max_samples=None, return_metrics=False, include_train=False, **kwargs):
    X_train, X_test, y_train, y_test = load_data(seed=seed, normalize=normalize, max_samples=max_samples)
    
    if model_type.lower() == "rf":
        model = train_random_forest(X_train, y_train, seed=seed, **kwargs)
    else:
        model = train_adaboost(X_train, y_train, seed=seed, **kwargs)
    
    metrics = evaluate(model, X_test, y_test, return_metrics=True)
    
    if include_train:
        metrics['train_accuracy'] = float(model.score(X_train, y_train))
        
    if return_metrics or include_train:
        return metrics
    return float(metrics["accuracy"])

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

O overfitting começa no momento em que a complexidade do modelo ultrapassa a necessidade do padrão dos dados. No caso das árvores de decisão:

O ponto de virada é a profundidade na qual a acurácia de teste para de subir (ou começa a cair), enquanto a de treino continua subindo em direção a 100%.

Identificação: Se o modelo fosse treinado variando a profundidade de 1 a 20, o ponto de overfitting seria o valor de `max_depth` que apresenta a maior acurácia no conjunto de teste.

Por que 100% no treino? Com `max_depth=None`, a árvore continua se expandindo até que todas as folhas sejam "puras" (contendo apenas uma classe). Como o Fashion MNIST possui padrões complexos, a árvore cria ramificações tão específicas que acabam descrevendo exemplos individuais do treino em vez de regras gerais da peça de roupa.

In [ ]:
depths = [2, 5, 10, 15, 20, None]
results = []

for d in depths:
    
    acc_data = run_pipeline(model_type="rf", max_depth=d, include_train=True)
    results.append({
        "depth": d if d is not None else "None",
        "train_acc": acc_data['train_accuracy'],
        "test_acc": acc_data['accuracy']
    })

import pandas as pd
print(pd.DataFrame(results))

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [6]:
import pandas as pd


def compare_models(seed=42, max_samples=None):

    rows = []
    for model_type in ["rf", "ab"]:
        metrics = run_pipeline(
            model_type=model_type,
            seed=seed,
            max_samples=max_samples,
            return_metrics=True,
        )
        metrics["model"] = "Random Forest" if model_type == "rf" else "AdaBoost"
        rows.append(metrics)

    result = pd.DataFrame(rows)[["model", "accuracy", "precision", "recall", "f1"]]
    return result.sort_values(by="f1", ascending=False).reset_index(drop=True)


def best_initial_model(seed=42, max_samples=None):
    table = compare_models(seed=seed, max_samples=max_samples)
    return table.iloc[0]["model"], table

**RESPOSTA - QUESTÃO 5**

**Resultados dos Modelos**

| Modelo        | Acurácia | Precisão | Recall | F1-Score |
| ------------- | -------- | -------- | ------ | -------- |
| Random Forest | 0.8710   | 0.8698   | 0.8710 | 0.8690   |
| AdaBoost      | 0.6433   | 0.6606   | 0.6433 | 0.6198   |

O **Random Forest** apresentou desempenho significativamente superior em todas as métricas avaliadas.

Ele obteve maior acurácia, além de melhores valores de precisão, recall e F1-score, indicando não apenas maior taxa de acertos gerais, mas também um equilíbrio mais consistente entre identificação correta das classes e redução de erros.

Já o AdaBoost teve um desempenho consideravelmente inferior, especialmente no F1-score, o que sugere menor equilíbrio entre precisão e recall.

Além disso, não necessariamente esse resultado se mantém com tuning de hiperparâmetros ou validação cruzada; às vezes o “melhor inicial” muda bastante ao se forçar os modelos a jogar em condições mais justas.


# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [7]:
def compare_seeds(seeds=(42, 7), model_types=("rf", "ab"), max_samples=None):

    rows = []
    for seed in seeds:
        for model_type in model_types:
            metrics = run_pipeline(
                model_type=model_type,
                seed=seed,
                max_samples=max_samples,
                return_metrics=True,
            )
            rows.append(
                {
                    "seed": seed,
                    "model": model_type,
                    "accuracy": metrics["accuracy"],
                    "precision": metrics["precision"],
                    "recall": metrics["recall"],
                    "f1": metrics["f1"],
                }
            )
    return pd.DataFrame(rows).sort_values(["model", "seed"]).reset_index(drop=True)


def reproducibility_check(model_type="rf", seed=42):
    acc1 = run_pipeline(model_type=model_type, seed=seed)
    acc2 = run_pipeline(model_type=model_type, seed=seed)
    return {
        "first_run": acc1,
        "second_run": acc2,
        "absolute_diff": abs(acc1 - acc2),
    }

**RESPOSTA - QUESTÃO 6**

Resultados com diferentes seeds

| Seed | Modelo        | Acurácia | Precisão | Recall | F1-Score |
| ---- | ------------- | -------- | -------- | ------ | -------- |
| 7    | AdaBoost      | 0.6760   | 0.6881   | 0.6760 | 0.6750   |
| 42   | AdaBoost      | 0.6433   | 0.6606   | 0.6433 | 0.6198   |
| 7    | Random Forest | 0.8547   | 0.8522   | 0.8547 | 0.8525   |
| 42   | Random Forest | 0.8710   | 0.8698   | 0.8710 | 0.8690   |


Sim, houve variação nos resultados ao alterar a seed.

Ao executar o pipeline com diferentes seeds (7 e 42), observa-se que os resultados variam para ambos os modelos, com mudanças mais perceptíveis no AdaBoost e variações mais suaves no Random Forest. Isso indica que fatores aleatórios, como a divisão entre treino e teste e processos internos dos algoritmos, influenciam o desempenho obtido. Dessa forma, o experimento pode ser considerado reprodutível apenas quando a mesma seed é mantida, garantindo a repetição exata dos resultados. No entanto, como há sensibilidade à escolha da seed, conclui-se que os resultados não são totalmente estáveis, sendo recomendável o uso de validação cruzada ou múltiplas execuções para obter uma avaliação mais confiável do desempenho dos modelos.


# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [8]:
def overfitting_report(seed=42, max_samples=None):

    X_train, X_test, y_train, y_test = load_data(seed=seed, max_samples=max_samples)

    rf = train_random_forest(X_train, y_train, seed=seed, n_estimators=250)
    ab = train_adaboost(X_train, y_train, seed=seed, n_estimators=120)

    rows = []
    for name, model in [("Random Forest", rf), ("AdaBoost", ab)]:
        train_acc = float(model.score(X_train, y_train))
        test_acc = float(model.score(X_test, y_test))
        rows.append(
            {
                "model": name,
                "train_accuracy": train_acc,
                "test_accuracy": test_acc,
                "generalization_gap": train_acc - test_acc,
            }
        )
    return pd.DataFrame(rows).sort_values("generalization_gap", ascending=False)

**RESPOSTA - QUESTÃO 7**

Sim, existe um overfitting acentuado, especialmente no modelo Random Forest. De acordo com os resultados obtidos na célula de análise de overfitting:

O Random Forest atingiu uma acurácia de 1.000 (100%) no treino, mas caiu para 0.871 no teste, gerando um generalization gap de aproximadamente 12.8%.

O AdaBoost apresentou um comportamento muito mais estável, com acurácia de 0.640 no treino e 0.637 no teste, resultando em um gap quase inexistente de 0.3%.

Qual modelo sofre mais?
O Random Forest sofre significativamente mais com overfitting neste cenário. Isso ocorre porque, com a configuração de max_depth=None, o modelo cria árvores extremamente profundas que "decoram" perfeitamente as instâncias de treino (incluindo ruídos), perdendo capacidade de generalização. O AdaBoost, por utilizar weak learners (árvores rasas), possui uma tendência natural menor ao overfitting severo em comparação a uma floresta de árvores sem restrição de profundidade.

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [9]:
def hyperparameter_sensitivity(seed=42, max_samples=None):

    rf_estimators = [50, 100, 200]
    ab_estimators = [30, 60, 120]

    rows = []
    for n in rf_estimators:
        metrics = run_pipeline(
            model_type="rf",
            seed=seed,
            max_samples=max_samples,
            return_metrics=True,
            n_estimators=n,
        )
        rows.append({"model": "rf", "n_estimators": n, **metrics})

    for n in ab_estimators:
        metrics = run_pipeline(
            model_type="ab",
            seed=seed,
            max_samples=max_samples,
            return_metrics=True,
            n_estimators=n,
        )
        rows.append({"model": "ab", "n_estimators": n, **metrics})

    return pd.DataFrame(rows).sort_values(["model", "n_estimators"]).reset_index(drop=True)

**RESPOSTA - QUESTÃO 8**

Analisando os resultados - ao variar o número de estimadores (`n_estimators`), o desempenho comportou-se de formas opostas entre os modelos:

- Random Forest: Mostrou-se estável e resiliente. A acurácia subiu de 0.866 (50 árvores) para 0.871 (200 árvores), indicando que o modelo converge para um platô positivo conforme mais estimadores são adicionados.
- AdaBoost: Mostrou-se altamente sensível e instável. O modelo atingiu seu melhor resultado com apenas 30 estimadores (0.686), mas o desempenho caiu drasticamente para 0.637 ao aumentar para 120 estimadores.

Sobre sensibilidade, nota-se que o AdaBoost é o modelo mais sensível. Isso acontece porque ele é um algoritmo de boosting sequencial, onde cada nova árvore tenta corrigir o erro da anterior. Com muitos estimadores e sem um ajuste fino da taxa de aprendizado (`learning_rate`), ele pode começar a focar excessivamente em exemplos atípicos (`outliers`) ou ruídos, degradando a performance global rapidamente. Já o Random Forest, por ser baseado em bagging (médias independentes), é naturalmente mais robusto a variações no número de estimadores.

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

1. **Acurácia**: A acurácia **NÃO** é suficiente para avaliar os modelos em datasets onde pode haver desbalanceamento ou onde o custo de falsos positivos/negativos difere. Utilizar apenas a acurácia global pode ocultar falhas do modelo em classes específicas ou ignorar o impacto de um possível desbalanceamento nos dados. Métricas como F1-Score e Matriz de Confusão oferecem uma visão mais granular do desempenho por classe; são, portanto, indispensáveis, pois oferecem uma visão qualitativa e detalhada, revelando se o desempenho é consistente em todas as categorias do Fashion MNIST.

2. **Reprodutibilidade**: A confiabilidade é estabelecida através do controle rigoroso da aleatoriedade, fixando o `random_state` em todas as etapas que envolvem sorteio de dados ou inicialização de algoritmos. Além disso, testar o pipeline com diferentes seeds funciona como um teste de sanidade, confirmando estatisticamente que os resultados são robustos e não fruto de uma divisão de dados excepcionalmente favorável.

3. **Problemas Metodológicos**:

- **Data Leakage**: Se a normalização fosse feita antes do split, informações do teste vazariam para o treino.

- **Falta de Validação Cruzada**: Basear a performance em apenas um split (treino/teste) pode levar a uma estimativa otimista ou pessimista demais dependendo da seed.

4. **Confiabilidade**: O pipeline possui uma confiabilidade parcial. Ele é bem estruturado no que diz respeito à modularidade e garante a reprodutibilidade dos experimentos através das sintonias de seed. Entretanto, para ter uma validação definitiva e ser considerado "robusto em produção", seria necessária a implementação de Validação Cruzada (Cross-Validation) e a separação de um conjunto de validação isolado, evitando que o ajuste de parâmetros comprometa a integridade da avaliação final.

## Resultados

In [11]:
print("="*50)
print("COMPARAÇÃO DE MODELOS - QUESTÕES 3, 4, 5, 8")
print("="*50)
df_models = compare_models()
print(df_models)

COMPARAÇÃO DE MODELOS - QUESTÕES 3, 4, 6, 8


,model,accuracy,precision,recall,f1
0,Random Forest,0.871000,0.869833,0.871000,0.868953
1,AdaBoost,0.643333,0.660635,0.643333,0.619798


In [12]:
print("\n" + "="*50)
print("COMPARAÇÃO DE DIFERENTES SEEDS - QUESTÃO 6")
print("="*50)
df_seeds = compare_seeds()
print(df_seeds)



COMPARAÇÃO DE DIFERENTES SEEDS - QUESTÃO 5


,seed,model,accuracy,precision,recall,f1
0,7,ab,0.676000,0.688144,0.676000,0.675010
1,42,ab,0.643333,0.660635,0.643333,0.619798
2,7,rf,0.854667,0.852223,0.854667,0.852452
3,42,rf,0.871000,0.869833,0.871000,0.868953


In [13]:
print("\n" + "="*50)
print("ANÁLISE DE OVERFITTING - RANDOM FOREST - QUESTÃO 7")
print("="*50)
df_overfitting = overfitting_report()
print(df_overfitting)


ANÁLISE DE OVERFITTING - RANDOM FOREST - QUESTÃO 7


,model,train_accuracy,test_accuracy,generalization_gap
0,Random Forest,1.000000,0.871333,0.128667
1,AdaBoost,0.640917,0.637000,0.003917


In [15]:
print("\n" + "="*50)
print("SENSIBILIDADE DE HIPERPARÂMETROS - ADABOOST - QUESTÃO 9")
print("="*50)
df_tuning = hyperparameter_sensitivity()
print(df_tuning)


SENSIBILIDADE DE HIPERPARÂMETROS - ADABOOST - QUESTÃO 9


,model,n_estimators,accuracy,precision,recall,f1
0,ab,30,0.686333,0.706372,0.686333,0.686175
1,ab,60,0.671333,0.687632,0.671333,0.667298
2,ab,120,0.637000,0.631830,0.637000,0.611298
3,rf,50,0.866000,0.864813,0.866000,0.863860
4,rf,100,0.870000,0.868903,0.870000,0.867566
5,rf,200,0.871000,0.869833,0.871000,0.868953
